In [1]:
# !python -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

!python -m pip install -U pandas matplotlib "cython<3.1" poetry-core
!python -m pip install --no-build-isolation python-woc

In [2]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = ['delftdata/valentine', 'darioizzo/geodesyNets', 'Antonio-JP/dd_functions',
            'ANYbotics/elevation_mapping', 'singmann/afex', 'opensafely/antibody-and-antiviral-deployment',
            'sebmart/TaxiSimulation', 'brainstorm-tools/brainstorm3', 'langurmonkey/gaiasky',
            'jump-dev/SumOfSquares.jl']
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

,sha1,project
0,00c5bbd0d4fdddc2a6f2ffec96b4a6472b4c7bef,delftdata_valentine


In [3]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

  6%|▌         | 137/2243 [02:26<38:15,  1.09s/it]

Got Errors {'088c72792e59048c961e350becd1d9abffa65fad': 'Key 088c72792e59048c961e350becd1d9abffa65fad not found in /da5_fast/All.sha1c/commit_8.tch'}


  6%|▋         | 144/2243 [02:33<37:39,  1.08s/it]

Got Errors {'1fb51ebb1f409dcb73a0b76ce609d058fc65e74c': 'Key 1fb51ebb1f409dcb73a0b76ce609d058fc65e74c not found in /da5_fast/All.sha1c/commit_31.tch'}


  7%|▋         | 147/2243 [02:37<37:34,  1.08s/it]

...

Got Errors {'e60a817afb958d9e9124d65ef7b4d11e6b26fe21': 'Key e60a817afb958d9e9124d65ef7b4d11e6b26fe21 not found in /da5_fast/All.sha1c/commit_102.tch'}


 88%|████████▊ | 1978/2243 [35:28<04:45,  1.08s/it]

Got Errors {'eee44114b5db59e99603450fe610ee8bce876eab': 'Key eee44114b5db59e99603450fe610ee8bce876eab not found in /da5_fast/All.sha1c/commit_110.tch'}


100%|██████████| 2243/2243 [40:14<00:00,  1.08s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,00c5bbd0d4fdddc2a6f2ffec96b4a6472b4c7bef,2c83dbede5978041b1842b7d0af5db7c15eee65c,[01807489e8d20b4b291770b72133f67b2ac8bf87],AndraIonescu <andradenis.ionescu@gmail.com>,1571149530,+0200,AndraIonescu <andradenis.ionescu@gmail.com>,1571149530,+0200,Added the final correlation clustering framewo...,00c5bbd0d4fdddc2a6f2ffec96b4a6472b4c7bef,delftdata_valentine
1,01807489e8d20b4b291770b72133f67b2ac8bf87,e581959571f850b0089cc9a4a183e40329ec9e74,[ba3da28481cfaf045b80a758c9ee9c5b376475a4],AndraIonescu <andradenis.ionescu@gmail.com>,1571142631,+0200,AndraIonescu <andradenis.ionescu@gmail.com>,1571142631,+0200,bug fixes\n,01807489e8d20b4b291770b72133f67b2ac8bf87,delftdata_valentine


In [4]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '00c5bbd0d4fdddc2a6f2ffec96b4a6472b4c7bef', 'tree': '2c83dbede5978041b1842b7d0af5db7c15eee65c', 'parent': ['01807489e8d20b4b291770b72133f67b2ac8bf87'], 'author': 'AndraIonescu <andradenis.ionescu@gmail.com>', 'author_time': 1571149530, 'author_tz': '+0200', 'committer': 'AndraIonescu <andradenis.ionescu@gmail.com>', 'committer_time': 1571149530, 'committer_tz': '+0200', 'message': 'Added the final correlation clustering framework\n'}


In [5]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = pd.DataFrame(columns=['project', 'commit', 'author', 'time', 'message'])
for k in commit_data:
  row = pd.Series({'project':prj, 'commit': k['commit'], 'author': k['author'], 'time': k['author_time'], 'message':k['message']})
  dfinf = pd.concat([dfinf, row.to_frame().T ], ignore_index=True)


In [6]:
# check if it has the right content
dfinf.head(1)

,project,commit,author,time,message
0,jump-dev_sumofsquares.jl,00c5bbd0d4fdddc2a6f2ffec96b4a6472b4c7bef,AndraIonescu <andradenis.ionescu@gmail.com>,1571149530,Added the final correlation clustering framewo...


In [7]:
#mode a means append, so you have all your projects in the same file
yournetid='dlong37'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='a', header=False)

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github